In [ ]:
#imports and environment
import csv
import json
import sys
import time

from collections import defaultdict
from datetime import datetime
from pathlib import Path

import torch #pytorch deep learning framework imports
import torch.nn.functional as torch_f

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import ( 
    AutoImageProcessor,
    DetrForObjectDetection,
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

c:\Users\megdo\Desktop\underwater-object-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: c:\Users\megdo\Desktop\underwater-object-detection\venv\Scripts\python.exe
PyTorch: 2.13.0+cpu
CUDA available: False


In [ ]:
#project and data paths
PROJECT_ROOT = Path(
    r"C:\Users\megdo\Desktop\underwater-object-detection"
)

ANNOTATION_DIR = ( # define the path for the annotation directory, which contains the COCO-style annotation files for the DUO dataset
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_coco"
    / "annotations"
)

TRAIN_IMAGE_DIR = ( # define the path for the training image directory, which contains the training images for the DUO dataset
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DUO"
    / "images"
    / "train"
)

TEST_IMAGE_DIR = ( # define the path for the test image directory, which contains the test images for the DUO dataset
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DUO"
    / "images"
    / "test"
)

TRAIN_JSON = ( 
    ANNOTATION_DIR
    / "instances_train_split.json"
)

VAL_JSON = (
    ANNOTATION_DIR
    / "instances_val_split.json"
)

TEST_JSON = (
    ANNOTATION_DIR
    / "instances_test_clean.json"
)

print("Project root:", PROJECT_ROOT.exists())
print("Training images:", TRAIN_IMAGE_DIR.exists())
print("Test images:", TEST_IMAGE_DIR.exists())
print("Training JSON:", TRAIN_JSON.exists())
print("Validation JSON:", VAL_JSON.exists())
print("Test JSON:", TEST_JSON.exists())

Project root: True
Training images: True
Test images: True
Training JSON: True
Validation JSON: True
Test JSON: True


In [ ]:
#labels and processor
MODEL_NAME = "facebook/detr-resnet-50"

id2label = { 
    0: "holothurian",
    1: "echinus",
    2: "scallop",
    3: "starfish",
}

label2id = {
    class_name: class_id
    for class_id, class_name in id2label.items()
}

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME
)

device = torch.device("cpu")

print("Processor loaded.")
print("Device:", device)
print("Classes:", id2label)

Processor loaded.
Device: cpu
Classes: {0: 'holothurian', 1: 'echinus', 2: 'scallop', 3: 'starfish'}


In [4]:
#model function
def create_fresh_detr_model():
    detr_model = (
        DetrForObjectDetection.from_pretrained(
            MODEL_NAME,
            num_labels=4,
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )
    )

    return detr_model.to(device)

In [ ]:
#dataset class
class DUODetrDataset(Dataset):
    def __init__(
        self,
        image_dir,
        annotation_file,
        processor,
    ):
        self.image_dir = Path(image_dir)
        self.processor = processor

        with Path(annotation_file).open(
            "r",
            encoding="utf-8",
        ) as file:
            self.coco = json.load(file)

        self.images = self.coco["images"]

        self.annotations_by_image = defaultdict(list)

        for annotation in self.coco["annotations"]:
            self.annotations_by_image[
                annotation["image_id"]
            ].append(annotation)

        category_ids = sorted(
            category["id"]
            for category in self.coco["categories"]
        )

        # Convert DUO category IDs 1–4 to DETR labels 0–3
        self.category_to_label = {
            category_id: index
            for index, category_id in enumerate(
                category_ids
            )
        }

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_record = self.images[index]

        image_path = (
            self.image_dir
            / image_record["file_name"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        annotations = []

        for annotation in self.annotations_by_image.get(
            image_record["id"],
            [],
        ):
            x, y, width, height = annotation["bbox"]

            if width <= 0 or height <= 0:
                continue

            annotations.append(
                {
                    "id": annotation["id"],
                    "image_id": image_record["id"],
                    "category_id":
                        self.category_to_label[
                            annotation["category_id"]
                        ],
                    "bbox": [
                        x,
                        y,
                        width,
                        height,
                    ],
                    "area": annotation.get(
                        "area",
                        width * height,
                    ),
                    "iscrowd": annotation.get(
                        "iscrowd",
                        0,
                    ),
                }
            )

        target = { 
            "image_id": image_record["id"],
            "annotations": annotations,
        }

        encoding = self.processor(
            images=image,
            annotations=target,
            return_tensors="pt",
            size={
                "shortest_edge": 416,
                "longest_edge": 416,
            },
        )

        pixel_values = encoding[
            "pixel_values"
        ].squeeze(0)

        labels = encoding["labels"][0]

        # These reshapes prevent the single-object error.
        labels["class_labels"] = labels[
            "class_labels"
        ].reshape(-1)

        labels["boxes"] = labels[
            "boxes"
        ].reshape(-1, 4)

        if "area" in labels:
            labels["area"] = labels[
                "area"
            ].reshape(-1)

        if "iscrowd" in labels:
            labels["iscrowd"] = labels[
                "iscrowd"
            ].reshape(-1)

        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }

In [6]:
#creating dataset
train_dataset = DUODetrDataset(
    TRAIN_IMAGE_DIR,
    TRAIN_JSON,
    processor,
)

val_dataset = DUODetrDataset(
    TRAIN_IMAGE_DIR,
    VAL_JSON,
    processor,
)

test_dataset = DUODetrDataset(
    TEST_IMAGE_DIR,
    TEST_JSON,
    processor,
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

Training images: 5336
Validation images: 1335
Test images: 1111


In [7]:
#manual padding function
def collate_fn(batch):
    pixel_values_list = [
        item["pixel_values"]
        for item in batch
    ]

    max_height = max(
        image.shape[1]
        for image in pixel_values_list
    )

    max_width = max(
        image.shape[2]
        for image in pixel_values_list
    )

    padded_images = []
    pixel_masks = []

    for image in pixel_values_list:
        _, height, width = image.shape

        pad_right = max_width - width
        pad_bottom = max_height - height

        padded_image = torch_f.pad(
            image,
            (
                0,
                pad_right,
                0,
                pad_bottom,
            ),
            value=0,
        )

        pixel_mask = torch.zeros(
            (max_height, max_width),
            dtype=torch.bool,
        )

        pixel_mask[:height, :width] = True

        padded_images.append(padded_image)
        pixel_masks.append(pixel_mask)

    return {
        "pixel_values": torch.stack(
            padded_images
        ),
        "pixel_mask": torch.stack(
            pixel_masks
        ),
        "labels": [
            item["labels"]
            for item in batch
        ],
    }

In [8]:
#data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 5336
Validation batches: 1335
Test batches: 1111


In [9]:
#testing batch
batch = next(iter(train_loader))

print(
    "Pixel batch:",
    batch["pixel_values"].shape,
)

print(
    "Labels shape:",
    batch["labels"][0][
        "class_labels"
    ].shape,
)

print(
    "Boxes shape:",
    batch["labels"][0][
        "boxes"
    ].shape,
)

Pixel batch: torch.Size([1, 3, 234, 416])
Labels shape: torch.Size([9])
Boxes shape: torch.Size([9, 4])


In [10]:
#testing an image
single_object_index = None

for index, image_record in enumerate(
    train_dataset.images
):
    image_id = image_record["id"]

    annotation_count = len(
        train_dataset.annotations_by_image.get(
            image_id,
            [],
        )
    )

    if annotation_count == 1:
        single_object_index = index
        break

print(
    "Single-object image index:",
    single_object_index,
)

single_sample = train_dataset[
    single_object_index
]

print(
    "Single-image labels shape:",
    single_sample["labels"][
        "class_labels"
    ].shape,
)

print(
    "Single-image boxes shape:",
    single_sample["labels"][
        "boxes"
    ].shape,
)

Single-object image index: 30
Single-image labels shape: torch.Size([1])
Single-image boxes shape: torch.Size([1, 4])


In [11]:
#forward pass test
test_model = create_fresh_detr_model()
test_model.train()

test_batch = next(iter(train_loader))

pixel_values = test_batch[
    "pixel_values"
].to(device)

pixel_mask = test_batch[
    "pixel_mask"
].to(device)

labels = [
    {
        key: value.to(device)
        if isinstance(value, torch.Tensor)
        else value
        for key, value in target.items()
    }
    for target in test_batch["labels"]
]

with torch.no_grad():
    outputs = test_model(
        pixel_values=pixel_values,
        pixel_mask=pixel_mask,
        labels=labels,
    )

print(
    "Forward-pass loss:",
    float(outputs.loss),
)

for loss_name, loss_value in (
    outputs.loss_dict.items()
):
    print(
        loss_name,
        float(loss_value),
    )

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `91`.
Loading weights: 100%|██████████| 530/530 [00:00<00:00, 5456.72it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |         

Forward-pass loss: 6.049570083618164
loss_ce 1.657971978187561
loss_bbox 0.43692636489868164
loss_giou 1.1034832000732422
cardinality_error 82.0


In [12]:
#deleting test model to free memory
del test_model
del outputs

model = create_fresh_detr_model()

print("Fresh full-training model created.")
print("Classes:", model.config.id2label)

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `91`.
Loading weights: 100%|██████████| 530/530 [00:00<00:00, 8044.98it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |         

Fresh full-training model created.
Classes: {0: 'holothurian', 1: 'echinus', 2: 'scallop', 3: 'starfish'}


In [13]:
#optimiser and scheduler
backbone_parameters = []
other_parameters = []

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if "backbone" in name:
        backbone_parameters.append(parameter)
    else:
        other_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {
            "params": other_parameters,
            "lr": 1e-4,
        },
        {
            "params": backbone_parameters,
            "lr": 1e-5,
        },
    ],
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1,
)

print(
    "Main learning rate:",
    optimizer.param_groups[0]["lr"],
)

print(
    "Backbone learning rate:",
    optimizer.param_groups[1]["lr"],
)

Main learning rate: 0.0001
Backbone learning rate: 1e-05


In [14]:
#results folder
RUN_DIR = (
    PROJECT_ROOT
    / "results"
    / "detr"
    / "duo_detr_resnet50_baseline_corrected"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LATEST_CHECKPOINT = (
    RUN_DIR
    / "latest_checkpoint.pth"
)

BEST_CHECKPOINT = (
    RUN_DIR
    / "best_checkpoint.pth"
)

HISTORY_CSV = (
    RUN_DIR
    / "training_history.csv"
)

print("Run directory:", RUN_DIR)
print("Latest checkpoint:", LATEST_CHECKPOINT)
print("Best checkpoint:", BEST_CHECKPOINT)

Run directory: C:\Users\megdo\Desktop\underwater-object-detection\results\detr\duo_detr_resnet50_baseline_corrected
Latest checkpoint: C:\Users\megdo\Desktop\underwater-object-detection\results\detr\duo_detr_resnet50_baseline_corrected\latest_checkpoint.pth
Best checkpoint: C:\Users\megdo\Desktop\underwater-object-detection\results\detr\duo_detr_resnet50_baseline_corrected\best_checkpoint.pth


In [15]:
#training function
def train_one_detr_epoch(
    model,
    data_loader,
    optimizer,
    device,
    epoch_number,
):
    model.train()

    epoch_start = time.perf_counter()
    total_epoch_loss = 0.0
    number_of_batches = len(data_loader)

    for batch_number, batch in enumerate(
        data_loader,
        start=1,
    ):
        pixel_values = batch[
            "pixel_values"
        ].to(device)

        pixel_mask = batch[
            "pixel_mask"
        ].to(device)

        labels = [
            {
                key: value.to(device)
                if isinstance(
                    value,
                    torch.Tensor,
                )
                else value
                for key, value in target.items()
            }
            for target in batch["labels"]
        ]

        outputs = model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
            labels=labels,
        )

        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss encountered: "
                f"{loss.item()}"
            )

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=0.1,
        )

        optimizer.step()

        batch_loss = float(
            loss.detach()
        )

        total_epoch_loss += batch_loss

        if (
            batch_number == 1
            or batch_number % 100 == 0
        ):
            elapsed_hours = (
                time.perf_counter()
                - epoch_start
            ) / 3600

            average_so_far = (
                total_epoch_loss
                / batch_number
            )

            print(
                f"Epoch {epoch_number} | "
                f"Batch {batch_number}/"
                f"{number_of_batches} | "
                f"Loss: {batch_loss:.4f} | "
                f"Average: "
                f"{average_so_far:.4f} | "
                f"Elapsed: "
                f"{elapsed_hours:.2f} hours"
            )

    epoch_seconds = (
        time.perf_counter()
        - epoch_start
    )

    average_train_loss = (
        total_epoch_loss
        / number_of_batches
    )

    return (
        average_train_loss,
        epoch_seconds,
    )

In [16]:
#validation function
def calculate_detr_validation_loss(
    model,
    data_loader,
    device,
):
    model.eval()

    validation_start = time.perf_counter()
    total_validation_loss = 0.0
    number_of_batches = len(data_loader)

    with torch.no_grad():
        for batch_number, batch in enumerate(
            data_loader,
            start=1,
        ):
            pixel_values = batch[
                "pixel_values"
            ].to(device)

            pixel_mask = batch[
                "pixel_mask"
            ].to(device)

            labels = [
                {
                    key: value.to(device)
                    if isinstance(
                        value,
                        torch.Tensor,
                    )
                    else value
                    for key, value
                    in target.items()
                }
                for target in batch["labels"]
            ]

            outputs = model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels,
            )

            total_validation_loss += float(
                outputs.loss.detach()
            )

            if (
                batch_number == 1
                or batch_number % 100 == 0
            ):
                print(
                    f"Validation batch "
                    f"{batch_number}/"
                    f"{number_of_batches}"
                )

    average_validation_loss = (
        total_validation_loss
        / number_of_batches
    )

    validation_seconds = (
        time.perf_counter()
        - validation_start
    )

    return (
        average_validation_loss,
        validation_seconds,
    )

In [17]:
#history and checkpoint saving functions
def append_detr_history(
    epoch_number,
    average_train_loss,
    average_validation_loss,
    epoch_seconds,
    validation_seconds,
    main_learning_rate,
    backbone_learning_rate,
):
    file_exists = HISTORY_CSV.exists()

    with HISTORY_CSV.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(
                [
                    "epoch",
                    "average_train_loss",
                    "average_validation_loss",
                    "training_seconds",
                    "training_hours",
                    "validation_seconds",
                    "main_learning_rate",
                    "backbone_learning_rate",
                    "completion_time",
                ]
            )

        writer.writerow(
            [
                epoch_number,
                average_train_loss,
                average_validation_loss,
                epoch_seconds,
                epoch_seconds / 3600,
                validation_seconds,
                main_learning_rate,
                backbone_learning_rate,
                datetime.now().isoformat(
                    timespec="seconds"
                ),
            ]
        )


def save_detr_checkpoint(
    epoch_number,
    average_train_loss,
    average_validation_loss,
    cumulative_training_seconds,
    best_validation_loss,
    is_best,
):
    checkpoint = {
        "completed_epoch": epoch_number,
        "model_state_dict":
            model.state_dict(),
        "optimizer_state_dict":
            optimizer.state_dict(),
        "scheduler_state_dict":
            scheduler.state_dict(),
        "average_train_loss":
            average_train_loss,
        "average_validation_loss":
            average_validation_loss,
        "best_validation_loss":
            best_validation_loss,
        "cumulative_training_seconds":
            cumulative_training_seconds,
        "id2label": id2label,
    }

    torch.save(
        checkpoint,
        LATEST_CHECKPOINT,
    )

    epoch_checkpoint = (
        RUN_DIR
        / f"checkpoint_epoch_"
          f"{epoch_number:02d}.pth"
    )

    torch.save(
        checkpoint,
        epoch_checkpoint,
    )

    if is_best:
        torch.save(
            checkpoint,
            BEST_CHECKPOINT,
        )

        print(
            "New best validation checkpoint saved."
        )

    print(
        "Latest checkpoint:",
        LATEST_CHECKPOINT,
    )

    print(
        "Epoch checkpoint:",
        epoch_checkpoint,
    )

In [18]:
#function to run and save epoch 1
def run_and_save_detr_epoch(
    epoch_number,
    cumulative_training_seconds,
    best_validation_loss,
):
    main_learning_rate = (
        optimizer.param_groups[0]["lr"]
    )

    backbone_learning_rate = (
        optimizer.param_groups[1]["lr"]
    )

    print("Beginning full DETR training")
    print("Epoch:", epoch_number)
    print(
        "Training images:",
        len(train_dataset),
    )
    print(
        "Main learning rate:",
        main_learning_rate,
    )
    print(
        "Backbone learning rate:",
        backbone_learning_rate,
    )
    print("Start time:", datetime.now())

    average_train_loss, epoch_seconds = (
        train_one_detr_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            device=device,
            epoch_number=epoch_number,
        )
    )

    average_validation_loss, validation_seconds = (
        calculate_detr_validation_loss(
            model=model,
            data_loader=val_loader,
            device=device,
        )
    )

    cumulative_training_seconds += (
        epoch_seconds
    )

    is_best = (
        average_validation_loss
        < best_validation_loss
    )

    if is_best:
        best_validation_loss = (
            average_validation_loss
        )

    scheduler.step()

    append_detr_history(
        epoch_number=epoch_number,
        average_train_loss=
            average_train_loss,
        average_validation_loss=
            average_validation_loss,
        epoch_seconds=epoch_seconds,
        validation_seconds=
            validation_seconds,
        main_learning_rate=
            main_learning_rate,
        backbone_learning_rate=
            backbone_learning_rate,
    )

    save_detr_checkpoint(
        epoch_number=epoch_number,
        average_train_loss=
            average_train_loss,
        average_validation_loss=
            average_validation_loss,
        cumulative_training_seconds=
            cumulative_training_seconds,
        best_validation_loss=
            best_validation_loss,
        is_best=is_best,
    )

    print("\nEpoch completed successfully.")
    print(
        f"Training loss: "
        f"{average_train_loss:.4f}"
    )
    print(
        f"Validation loss: "
        f"{average_validation_loss:.4f}"
    )
    print(
        f"Training time: "
        f"{epoch_seconds / 3600:.2f} hours"
    )
    print(
        f"Validation time: "
        f"{validation_seconds / 3600:.2f} hours"
    )
    print(
        "Cumulative training time:",
        f"{cumulative_training_seconds / 3600:.2f} hours",
    )
    print("Finish time:", datetime.now())

    return (
        cumulative_training_seconds,
        best_validation_loss,
    )

In [19]:
#start epoch 1
epoch_number = 1
cumulative_training_seconds = 0.0
best_validation_loss = float("inf")

(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 1
Training images: 5336
Main learning rate: 0.0001
Backbone learning rate: 1e-05
Start time: 2026-07-22 12:09:17.660432
Epoch 1 | Batch 1/5336 | Loss: 10.4199 | Average: 10.4199 | Elapsed: 0.00 hours
Epoch 1 | Batch 100/5336 | Loss: 3.5155 | Average: 3.4317 | Elapsed: 0.02 hours
Epoch 1 | Batch 200/5336 | Loss: 2.3880 | Average: 3.1962 | Elapsed: 0.04 hours
Epoch 1 | Batch 300/5336 | Loss: 2.3089 | Average: 3.0826 | Elapsed: 0.06 hours
Epoch 1 | Batch 400/5336 | Loss: 2.1083 | Average: 3.0410 | Elapsed: 0.08 hours
Epoch 1 | Batch 500/5336 | Loss: 3.1797 | Average: 3.0326 | Elapsed: 0.10 hours
Epoch 1 | Batch 600/5336 | Loss: 3.5371 | Average: 2.9862 | Elapsed: 0.12 hours
Epoch 1 | Batch 700/5336 | Loss: 4.2355 | Average: 2.9475 | Elapsed: 0.14 hours
Epoch 1 | Batch 800/5336 | Loss: 2.4580 | Average: 2.9347 | Elapsed: 0.16 hours
Epoch 1 | Batch 900/5336 | Loss: 2.5950 | Average: 2.8798 | Elapsed: 0.18 hours
Epoch 1 | Batch 1000/5336 | Loss: 4.4715 | A

In [20]:
#epcoh 1 checkpoint
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print(
    "Best validation loss:",
    best_validation_loss,
)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)

Completed epoch: 1
Next epoch: 2
Cumulative training time: 1.03 hours
Best validation loss: 2.356624818372816
Current main learning rate: 0.0001


In [21]:
#run epoch 2
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 2
Training images: 5336
Main learning rate: 0.0001
Backbone learning rate: 1e-05
Start time: 2026-07-22 13:45:58.302490
Epoch 2 | Batch 1/5336 | Loss: 1.8504 | Average: 1.8504 | Elapsed: 0.00 hours
Epoch 2 | Batch 100/5336 | Loss: 1.8357 | Average: 2.4874 | Elapsed: 0.03 hours
Epoch 2 | Batch 200/5336 | Loss: 2.8583 | Average: 2.4280 | Elapsed: 0.06 hours
Epoch 2 | Batch 300/5336 | Loss: 2.1305 | Average: 2.3986 | Elapsed: 0.09 hours
Epoch 2 | Batch 400/5336 | Loss: 2.8391 | Average: 2.3590 | Elapsed: 0.28 hours
Epoch 2 | Batch 500/5336 | Loss: 1.1373 | Average: 2.3803 | Elapsed: 0.34 hours
Epoch 2 | Batch 600/5336 | Loss: 3.6678 | Average: 2.3661 | Elapsed: 0.39 hours
Epoch 2 | Batch 700/5336 | Loss: 2.5214 | Average: 2.3461 | Elapsed: 0.42 hours
Epoch 2 | Batch 800/5336 | Loss: 4.9134 | Average: 2.3337 | Elapsed: 0.47 hours
Epoch 2 | Batch 900/5336 | Loss: 2.8176 | Average: 2.3328 | Elapsed: 0.53 hours
Epoch 2 | Batch 1000/5336 | Loss: 2.2768 | Ave

In [22]:
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Best validation loss:",
    best_validation_loss,
)

Completed epoch: 2
Next epoch: 3
Best validation loss: 1.8882443545211791


In [23]:
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 3
Training images: 5336
Main learning rate: 0.0001
Backbone learning rate: 1e-05
Start time: 2026-07-22 16:35:16.181905
Epoch 3 | Batch 1/5336 | Loss: 2.7265 | Average: 2.7265 | Elapsed: 0.00 hours
Epoch 3 | Batch 100/5336 | Loss: 1.7876 | Average: 1.8228 | Elapsed: 0.02 hours
Epoch 3 | Batch 200/5336 | Loss: 1.9521 | Average: 1.8974 | Elapsed: 0.04 hours
Epoch 3 | Batch 300/5336 | Loss: 2.1659 | Average: 1.9126 | Elapsed: 0.06 hours
Epoch 3 | Batch 400/5336 | Loss: 1.6199 | Average: 1.9125 | Elapsed: 0.08 hours
Epoch 3 | Batch 500/5336 | Loss: 2.1522 | Average: 1.9083 | Elapsed: 0.10 hours
Epoch 3 | Batch 600/5336 | Loss: 1.7431 | Average: 1.9084 | Elapsed: 0.12 hours
Epoch 3 | Batch 700/5336 | Loss: 2.0608 | Average: 1.9192 | Elapsed: 0.14 hours
Epoch 3 | Batch 800/5336 | Loss: 1.5608 | Average: 1.9206 | Elapsed: 0.16 hours
Epoch 3 | Batch 900/5336 | Loss: 0.6836 | Average: 1.9101 | Elapsed: 0.19 hours
Epoch 3 | Batch 1000/5336 | Loss: 1.6990 | Ave

In [24]:
#checkpoint after epoch 3
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Best validation loss:",
    best_validation_loss,
)

Completed epoch: 3
Next epoch: 4
Best validation loss: 1.697843568013067


In [25]:
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 4
Training images: 5336
Main learning rate: 0.0001
Backbone learning rate: 1e-05
Start time: 2026-07-22 17:51:35.953846
Epoch 4 | Batch 1/5336 | Loss: 2.5143 | Average: 2.5143 | Elapsed: 0.00 hours
Epoch 4 | Batch 100/5336 | Loss: 1.1835 | Average: 1.7998 | Elapsed: 0.02 hours
Epoch 4 | Batch 200/5336 | Loss: 1.7139 | Average: 1.7286 | Elapsed: 0.04 hours
Epoch 4 | Batch 300/5336 | Loss: 2.0769 | Average: 1.7110 | Elapsed: 0.06 hours
Epoch 4 | Batch 400/5336 | Loss: 1.6328 | Average: 1.7068 | Elapsed: 0.08 hours
Epoch 4 | Batch 500/5336 | Loss: 1.7795 | Average: 1.7211 | Elapsed: 0.10 hours
Epoch 4 | Batch 600/5336 | Loss: 1.2618 | Average: 1.7328 | Elapsed: 0.12 hours
Epoch 4 | Batch 700/5336 | Loss: 2.6413 | Average: 1.7320 | Elapsed: 0.14 hours
Epoch 4 | Batch 800/5336 | Loss: 1.0082 | Average: 1.7299 | Elapsed: 0.17 hours
Epoch 4 | Batch 900/5336 | Loss: 2.1939 | Average: 1.7274 | Elapsed: 0.19 hours
Epoch 4 | Batch 1000/5336 | Loss: 1.0412 | Ave

In [26]:
#loading checkpoint after epoch 4
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)

Completed epoch: 4
Next epoch: 5
Best validation loss: 1.697843568013067
Current main learning rate: 0.0001


In [27]:
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

print("Completed epoch:", checkpoint["completed_epoch"])

Completed epoch: 4


In [28]:
#redoing epoch 5 after scheduler step
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)

Completed epoch: 4
Next epoch: 5
Current main learning rate: 0.0001


In [29]:
#restoring epoch 5 (didnt finish last time)
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 5
Training images: 5336
Main learning rate: 0.0001
Backbone learning rate: 1e-05
Start time: 2026-07-22 20:32:56.066685
Epoch 5 | Batch 1/5336 | Loss: 1.9522 | Average: 1.9522 | Elapsed: 0.00 hours
Epoch 5 | Batch 100/5336 | Loss: 1.2809 | Average: 1.6251 | Elapsed: 0.04 hours
Epoch 5 | Batch 200/5336 | Loss: 2.0752 | Average: 1.6173 | Elapsed: 0.09 hours
Epoch 5 | Batch 300/5336 | Loss: 1.2431 | Average: 1.6379 | Elapsed: 0.13 hours
Epoch 5 | Batch 400/5336 | Loss: 1.2244 | Average: 1.6690 | Elapsed: 0.17 hours
Epoch 5 | Batch 500/5336 | Loss: 0.5340 | Average: 1.6584 | Elapsed: 1.46 hours
Epoch 5 | Batch 600/5336 | Loss: 1.4680 | Average: 1.6415 | Elapsed: 1.52 hours
Epoch 5 | Batch 700/5336 | Loss: 1.7133 | Average: 1.6407 | Elapsed: 1.57 hours
Epoch 5 | Batch 800/5336 | Loss: 2.6347 | Average: 1.6432 | Elapsed: 1.59 hours
Epoch 5 | Batch 900/5336 | Loss: 2.3923 | Average: 1.6590 | Elapsed: 1.62 hours
Epoch 5 | Batch 1000/5336 | Loss: 1.7493 | Ave

In [30]:
#latest checkpoint after epoch 5
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)
print(
    "Current backbone learning rate:",
    optimizer.param_groups[1]["lr"],
)

Completed epoch: 5
Next epoch: 6
Current main learning rate: 1e-05
Current backbone learning rate: 1.0000000000000002e-06


In [31]:
#starting epoch 6
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 6
Training images: 5336
Main learning rate: 1e-05
Backbone learning rate: 1.0000000000000002e-06
Start time: 2026-07-23 11:16:09.701885
Epoch 6 | Batch 1/5336 | Loss: 1.1957 | Average: 1.1957 | Elapsed: 0.00 hours
Epoch 6 | Batch 100/5336 | Loss: 2.0740 | Average: 1.6537 | Elapsed: 0.03 hours
Epoch 6 | Batch 200/5336 | Loss: 3.0102 | Average: 1.6054 | Elapsed: 0.06 hours
Epoch 6 | Batch 300/5336 | Loss: 0.9002 | Average: 1.5812 | Elapsed: 0.08 hours
Epoch 6 | Batch 400/5336 | Loss: 2.0233 | Average: 1.5471 | Elapsed: 0.10 hours
Epoch 6 | Batch 500/5336 | Loss: 3.3671 | Average: 1.5343 | Elapsed: 0.12 hours
Epoch 6 | Batch 600/5336 | Loss: 1.4019 | Average: 1.5193 | Elapsed: 0.14 hours
Epoch 6 | Batch 700/5336 | Loss: 0.4578 | Average: 1.5003 | Elapsed: 0.16 hours
Epoch 6 | Batch 800/5336 | Loss: 0.9191 | Average: 1.4945 | Elapsed: 0.19 hours
Epoch 6 | Batch 900/5336 | Loss: 1.5022 | Average: 1.4878 | Elapsed: 0.21 hours
Epoch 6 | Batch 1000/5336 | Lo

In [32]:
import pandas as pd

history_path = RUN_DIR / "training_history.csv"
history_df = pd.read_csv(history_path)

display(history_df)

,epoch,average_train_loss,average_validation_loss,training_seconds,training_hours,validation_seconds,main_learning_rate,backbone_learning_rate,completion_time
0,1,2.669919,2.356625,3720.928979,1.033591,251.220950,0.00010,0.000010,2026-07-22T13:15:29
1,2,2.142196,1.888244,8788.364654,2.441212,251.585416,0.00010,0.000010,2026-07-22T16:16:38
2,3,1.880384,1.697844,3940.074978,1.094465,341.408232,0.00010,0.000010,2026-07-22T17:46:37
3,4,1.740071,1.755049,3999.611199,1.111003,272.069183,0.00010,0.000010,2026-07-22T19:02:47
4,5,1.661423,1.634695,52240.026700,14.511119,248.149900,0.00010,0.000010,2026-07-23T11:07:41
5,6,1.368582,1.287403,4173.666068,1.159352,249.223950,0.00001,0.000001,2026-07-23T12:29:52


In [33]:
#loading latest checkpoint after epoch 6
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)
print(
    "Current backbone learning rate:",
    optimizer.param_groups[1]["lr"],
)

Completed epoch: 6
Next epoch: 7
Best validation loss: 1.2874028298700375
Current main learning rate: 1e-05
Current backbone learning rate: 1.0000000000000002e-06


In [34]:
#epoch 7
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 7
Training images: 5336
Main learning rate: 1e-05
Backbone learning rate: 1.0000000000000002e-06
Start time: 2026-07-23 12:56:13.901393
Epoch 7 | Batch 1/5336 | Loss: 0.4309 | Average: 0.4309 | Elapsed: 0.00 hours
Epoch 7 | Batch 100/5336 | Loss: 0.9478 | Average: 1.1397 | Elapsed: 0.03 hours
Epoch 7 | Batch 200/5336 | Loss: 1.6721 | Average: 1.2698 | Elapsed: 0.06 hours
Epoch 7 | Batch 300/5336 | Loss: 1.6443 | Average: 1.2821 | Elapsed: 0.09 hours
Epoch 7 | Batch 400/5336 | Loss: 2.0230 | Average: 1.2996 | Elapsed: 0.14 hours
Epoch 7 | Batch 500/5336 | Loss: 0.6210 | Average: 1.2926 | Elapsed: 0.20 hours
Epoch 7 | Batch 600/5336 | Loss: 1.3958 | Average: 1.2839 | Elapsed: 0.26 hours
Epoch 7 | Batch 700/5336 | Loss: 1.1783 | Average: 1.2830 | Elapsed: 0.32 hours
Epoch 7 | Batch 800/5336 | Loss: 0.8173 | Average: 1.2983 | Elapsed: 0.38 hours
Epoch 7 | Batch 900/5336 | Loss: 1.1696 | Average: 1.2925 | Elapsed: 0.42 hours
Epoch 7 | Batch 1000/5336 | Lo

In [35]:
history_df = pd.read_csv(
    RUN_DIR / "training_history.csv"
)

display(
    history_df[
        [
            "epoch",
            "average_train_loss",
            "average_validation_loss",
            "training_hours",
            "main_learning_rate",
        ]
    ]
)

,epoch,average_train_loss,average_validation_loss,training_hours,main_learning_rate
0,1,2.669919,2.356625,1.033591,0.00010
1,2,2.142196,1.888244,2.441212,0.00010
2,3,1.880384,1.697844,1.094465,0.00010
3,4,1.740071,1.755049,1.111003,0.00010
4,5,1.661423,1.634695,14.511119,0.00010
5,6,1.368582,1.287403,1.159352,0.00001
6,7,1.269332,1.232705,2.309147,0.00001


In [36]:
#reloading latest checkpoint after epoch 7
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)

Completed epoch: 7
Next epoch: 8
Best validation loss: 1.232704552378009


In [37]:
#epoch 8
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 8
Training images: 5336
Main learning rate: 1e-05
Backbone learning rate: 1.0000000000000002e-06
Start time: 2026-07-23 15:26:29.534704
Epoch 8 | Batch 1/5336 | Loss: 0.8703 | Average: 0.8703 | Elapsed: 0.00 hours
Epoch 8 | Batch 100/5336 | Loss: 0.8844 | Average: 1.1880 | Elapsed: 0.04 hours
Epoch 8 | Batch 200/5336 | Loss: 2.0416 | Average: 1.1862 | Elapsed: 0.07 hours
Epoch 8 | Batch 300/5336 | Loss: 0.5755 | Average: 1.2435 | Elapsed: 0.11 hours
Epoch 8 | Batch 400/5336 | Loss: 1.6868 | Average: 1.2365 | Elapsed: 0.16 hours
Epoch 8 | Batch 500/5336 | Loss: 2.2168 | Average: 1.2516 | Elapsed: 0.20 hours
Epoch 8 | Batch 600/5336 | Loss: 1.5636 | Average: 1.2475 | Elapsed: 0.26 hours
Epoch 8 | Batch 700/5336 | Loss: 2.2944 | Average: 1.2467 | Elapsed: 0.31 hours
Epoch 8 | Batch 800/5336 | Loss: 2.2687 | Average: 1.2505 | Elapsed: 0.36 hours
Epoch 8 | Batch 900/5336 | Loss: 1.3878 | Average: 1.2477 | Elapsed: 0.41 hours
Epoch 8 | Batch 1000/5336 | Lo

In [38]:
#checking validation loss after epoch 8
history_df = pd.read_csv(
    RUN_DIR / "training_history.csv"
)

display(
    history_df[
        [
            "epoch",
            "average_train_loss",
            "average_validation_loss",
            "training_hours",
            "main_learning_rate",
        ]
    ]
)

,epoch,average_train_loss,average_validation_loss,training_hours,main_learning_rate
0,1,2.669919,2.356625,1.033591,0.00010
1,2,2.142196,1.888244,2.441212,0.00010
2,3,1.880384,1.697844,1.094465,0.00010
3,4,1.740071,1.755049,1.111003,0.00010
4,5,1.661423,1.634695,14.511119,0.00010
5,6,1.368582,1.287403,1.159352,0.00001
6,7,1.269332,1.232705,2.309147,0.00001
7,8,1.217745,1.201944,1.841752,0.00001


In [39]:
#checkpoint 
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)

Completed epoch: 8
Next epoch: 9
Best validation loss: 1.2019440689584378


In [41]:
#run epoch 9
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 9
Training images: 5336
Main learning rate: 1e-05
Backbone learning rate: 1.0000000000000002e-06
Start time: 2026-07-23 17:33:29.229277
Epoch 9 | Batch 1/5336 | Loss: 0.8521 | Average: 0.8521 | Elapsed: 0.00 hours
Epoch 9 | Batch 100/5336 | Loss: 0.9787 | Average: 1.1494 | Elapsed: 0.03 hours
Epoch 9 | Batch 200/5336 | Loss: 1.2988 | Average: 1.2047 | Elapsed: 0.06 hours
Epoch 9 | Batch 300/5336 | Loss: 2.1992 | Average: 1.1527 | Elapsed: 0.10 hours
Epoch 9 | Batch 400/5336 | Loss: 1.6895 | Average: 1.1497 | Elapsed: 0.14 hours
Epoch 9 | Batch 500/5336 | Loss: 1.0909 | Average: 1.1543 | Elapsed: 0.18 hours
Epoch 9 | Batch 600/5336 | Loss: 0.8409 | Average: 1.1595 | Elapsed: 0.23 hours
Epoch 9 | Batch 700/5336 | Loss: 1.4754 | Average: 1.1660 | Elapsed: 0.27 hours
Epoch 9 | Batch 800/5336 | Loss: 0.5008 | Average: 1.1557 | Elapsed: 0.32 hours
Epoch 9 | Batch 900/5336 | Loss: 1.0146 | Average: 1.1646 | Elapsed: 0.36 hours
Epoch 9 | Batch 1000/5336 | Lo

In [42]:
history_df = pd.read_csv(
    RUN_DIR / "training_history.csv"
)

display(
    history_df[
        [
            "epoch",
            "average_train_loss",
            "average_validation_loss",
            "training_hours",
            "main_learning_rate",
        ]
    ]
)

,epoch,average_train_loss,average_validation_loss,training_hours,main_learning_rate
0,1,2.669919,2.356625,1.033591,0.00010
1,2,2.142196,1.888244,2.441212,0.00010
2,3,1.880384,1.697844,1.094465,0.00010
3,4,1.740071,1.755049,1.111003,0.00010
4,5,1.661423,1.634695,14.511119,0.00010
5,6,1.368582,1.287403,1.159352,0.00001
6,7,1.269332,1.232705,2.309147,0.00001
7,8,1.217745,1.201944,1.841752,0.00001
8,9,1.175973,1.178760,1.783136,0.00001


In [43]:
#checkpoint
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)

Completed epoch: 9
Next epoch: 10
Best validation loss: 1.178760255537283
Current main learning rate: 1e-05


In [44]:
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 10
Training images: 5336
Main learning rate: 1e-05
Backbone learning rate: 1.0000000000000002e-06
Start time: 2026-07-23 20:53:24.910445
Epoch 10 | Batch 1/5336 | Loss: 0.7182 | Average: 0.7182 | Elapsed: 0.00 hours
Epoch 10 | Batch 100/5336 | Loss: 1.3853 | Average: 1.1300 | Elapsed: 0.04 hours
Epoch 10 | Batch 200/5336 | Loss: 1.2820 | Average: 1.1367 | Elapsed: 0.08 hours
Epoch 10 | Batch 300/5336 | Loss: 0.5195 | Average: 1.1179 | Elapsed: 0.13 hours
Epoch 10 | Batch 400/5336 | Loss: 1.3569 | Average: 1.1369 | Elapsed: 0.17 hours
Epoch 10 | Batch 500/5336 | Loss: 0.8331 | Average: 1.1388 | Elapsed: 0.21 hours
Epoch 10 | Batch 600/5336 | Loss: 1.2950 | Average: 1.1421 | Elapsed: 0.25 hours
Epoch 10 | Batch 700/5336 | Loss: 2.0845 | Average: 1.1385 | Elapsed: 0.30 hours
Epoch 10 | Batch 800/5336 | Loss: 2.8576 | Average: 1.1395 | Elapsed: 0.34 hours
Epoch 10 | Batch 900/5336 | Loss: 1.7603 | Average: 1.1383 | Elapsed: 0.38 hours
Epoch 10 | Batch 10

In [45]:
history_df = pd.read_csv(
    RUN_DIR / "training_history.csv"
)

display(
    history_df[
        [
            "epoch",
            "average_train_loss",
            "average_validation_loss",
            "training_hours",
            "main_learning_rate",
        ]
    ]
)

,epoch,average_train_loss,average_validation_loss,training_hours,main_learning_rate
0,1,2.669919,2.356625,1.033591,0.00010
1,2,2.142196,1.888244,2.441212,0.00010
2,3,1.880384,1.697844,1.094465,0.00010
3,4,1.740071,1.755049,1.111003,0.00010
4,5,1.661423,1.634695,14.511119,0.00010
5,6,1.368582,1.287403,1.159352,0.00001
6,7,1.269332,1.232705,2.309147,0.00001
7,8,1.217745,1.201944,1.841752,0.00001
8,9,1.175973,1.178760,1.783136,0.00001
9,10,1.148679,1.157855,16.230681,0.00001


In [46]:
#loading latest checkpoint
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)
print(
    "Current backbone learning rate:",
    optimizer.param_groups[1]["lr"],
)

Completed epoch: 10
Next epoch: 11
Best validation loss: 1.157855187086741
Current main learning rate: 1.0000000000000002e-06
Current backbone learning rate: 1.0000000000000002e-07


In [47]:
#epoch 11
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 11
Training images: 5336
Main learning rate: 1.0000000000000002e-06
Backbone learning rate: 1.0000000000000002e-07
Start time: 2026-07-24 13:24:52.185984
Epoch 11 | Batch 1/5336 | Loss: 1.0218 | Average: 1.0218 | Elapsed: 0.00 hours
Epoch 11 | Batch 100/5336 | Loss: 1.2771 | Average: 1.1469 | Elapsed: 0.03 hours
Epoch 11 | Batch 200/5336 | Loss: 1.7571 | Average: 1.1479 | Elapsed: 0.06 hours
Epoch 11 | Batch 300/5336 | Loss: 0.7494 | Average: 1.1486 | Elapsed: 0.11 hours
Epoch 11 | Batch 400/5336 | Loss: 1.6522 | Average: 1.1305 | Elapsed: 0.16 hours
Epoch 11 | Batch 500/5336 | Loss: 0.8874 | Average: 1.1428 | Elapsed: 0.21 hours
Epoch 11 | Batch 600/5336 | Loss: 0.7718 | Average: 1.1313 | Elapsed: 0.26 hours
Epoch 11 | Batch 700/5336 | Loss: 0.7197 | Average: 1.1303 | Elapsed: 0.31 hours
Epoch 11 | Batch 800/5336 | Loss: 0.5704 | Average: 1.1204 | Elapsed: 0.36 hours
Epoch 11 | Batch 900/5336 | Loss: 0.8236 | Average: 1.1160 | Elapsed: 0.41 hours
Ep

In [48]:
#epoch 11 validation loss check
history_df = pd.read_csv(
    RUN_DIR / "training_history.csv"
)

display(
    history_df[
        [
            "epoch",
            "average_train_loss",
            "average_validation_loss",
            "training_hours",
            "main_learning_rate",
        ]
    ]
)

,epoch,average_train_loss,average_validation_loss,training_hours,main_learning_rate
0,1,2.669919,2.356625,1.033591,0.000100
1,2,2.142196,1.888244,2.441212,0.000100
2,3,1.880384,1.697844,1.094465,0.000100
3,4,1.740071,1.755049,1.111003,0.000100
4,5,1.661423,1.634695,14.511119,0.000100
5,6,1.368582,1.287403,1.159352,0.000010
6,7,1.269332,1.232705,2.309147,0.000010
7,8,1.217745,1.201944,1.841752,0.000010
8,9,1.175973,1.178760,1.783136,0.000010
9,10,1.148679,1.157855,16.230681,0.000010


In [49]:
#checkpoint
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]
epoch_number = completed_epoch + 1

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

best_validation_loss = checkpoint[
    "best_validation_loss"
]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print("Best validation loss:", best_validation_loss)
print(
    "Current main learning rate:",
    optimizer.param_groups[0]["lr"],
)

Completed epoch: 11
Next epoch: 12
Best validation loss: 1.1462732903545463
Current main learning rate: 1.0000000000000002e-06


In [50]:
#epoch 12
(
    cumulative_training_seconds,
    best_validation_loss,
) = run_and_save_detr_epoch(
    epoch_number=epoch_number,
    cumulative_training_seconds=
        cumulative_training_seconds,
    best_validation_loss=
        best_validation_loss,
)

Beginning full DETR training
Epoch: 12
Training images: 5336
Main learning rate: 1.0000000000000002e-06
Backbone learning rate: 1.0000000000000002e-07
Start time: 2026-07-24 15:41:58.711047
Epoch 12 | Batch 1/5336 | Loss: 2.0698 | Average: 2.0698 | Elapsed: 0.00 hours
Epoch 12 | Batch 100/5336 | Loss: 0.4486 | Average: 1.1094 | Elapsed: 0.02 hours
Epoch 12 | Batch 200/5336 | Loss: 0.7446 | Average: 1.1045 | Elapsed: 0.04 hours
Epoch 12 | Batch 300/5336 | Loss: 0.8471 | Average: 1.0845 | Elapsed: 0.07 hours
Epoch 12 | Batch 400/5336 | Loss: 1.2383 | Average: 1.1159 | Elapsed: 0.09 hours
Epoch 12 | Batch 500/5336 | Loss: 2.3643 | Average: 1.0977 | Elapsed: 0.11 hours
Epoch 12 | Batch 600/5336 | Loss: 0.5184 | Average: 1.1076 | Elapsed: 0.14 hours
Epoch 12 | Batch 700/5336 | Loss: 0.8958 | Average: 1.1125 | Elapsed: 0.16 hours
Epoch 12 | Batch 800/5336 | Loss: 0.5239 | Average: 1.1225 | Elapsed: 0.18 hours
Epoch 12 | Batch 900/5336 | Loss: 0.6820 | Average: 1.1169 | Elapsed: 0.21 hours
Ep

In [51]:
history_df = pd.read_csv(
    RUN_DIR / "training_history.csv"
)

display(
    history_df[
        [
            "epoch",
            "average_train_loss",
            "average_validation_loss",
            "training_hours",
            "main_learning_rate",
        ]
    ]
)

,epoch,average_train_loss,average_validation_loss,training_hours,main_learning_rate
0,1,2.669919,2.356625,1.033591,0.000100
1,2,2.142196,1.888244,2.441212,0.000100
2,3,1.880384,1.697844,1.094465,0.000100
3,4,1.740071,1.755049,1.111003,0.000100
4,5,1.661423,1.634695,14.511119,0.000100
5,6,1.368582,1.287403,1.159352,0.000010
6,7,1.269332,1.232705,2.309147,0.000010
7,8,1.217745,1.201944,1.841752,0.000010
8,9,1.175973,1.178760,1.783136,0.000010
9,10,1.148679,1.157855,16.230681,0.000010


In [52]:
#best checkpoint
best_checkpoint_path = RUN_DIR / "best_checkpoint.pth"

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Loaded best epoch:", checkpoint["completed_epoch"])
print("Best validation loss:", checkpoint["best_validation_loss"])

Loaded best epoch: 12
Best validation loss: 1.1431484567980936


In [53]:
%pip install torchmetrics pycocotools

Note: you may need to restart the kernel to use updated packages.


In [54]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

test_metric = MeanAveragePrecision(
    box_format="xyxy",
    iou_type="bbox",
    class_metrics=True,
)

In [56]:
#evaluation cell
import time
import torch

model.eval()
test_metric.reset()

total_inference_seconds = 0.0
total_images = 0

with torch.no_grad():
    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        labels = batch["labels"]

        start_time = time.perf_counter()

        outputs = model(
            pixel_values=pixel_values,
            pixel_mask=pixel_mask,
        )

        total_inference_seconds += (
            time.perf_counter() - start_time
        )

        target_sizes = torch.stack(
            [
                label["orig_size"]
                for label in labels
            ]
        ).to(device)

        predictions = (
            processor.post_process_object_detection(
                outputs,
                target_sizes=target_sizes,
                threshold=0.0,
            )
        )

        metric_predictions = []
        metric_targets = []

        for prediction, label in zip(
            predictions,
            labels,
        ):
            metric_predictions.append(
                {
                    "boxes": prediction["boxes"].cpu(),
                    "scores": prediction["scores"].cpu(),
                    "labels": prediction["labels"].cpu(),
                }
            )

            image_height = label["orig_size"][0]
            image_width = label["orig_size"][1]

            boxes = label["boxes"].clone()

            centre_x = boxes[:, 0]
            centre_y = boxes[:, 1]
            box_width = boxes[:, 2]
            box_height = boxes[:, 3]

            x_min = (
                centre_x - box_width / 2
            ) * image_width

            y_min = (
                centre_y - box_height / 2
            ) * image_height

            x_max = (
                centre_x + box_width / 2
            ) * image_width

            y_max = (
                centre_y + box_height / 2
            ) * image_height

            target_boxes = torch.stack(
                [x_min, y_min, x_max, y_max],
                dim=1,
            )

            metric_targets.append(
                {
                    "boxes": target_boxes.cpu(),
                    "labels": label[
                        "class_labels"
                    ].cpu(),
                }
            )

        test_metric.update(
            metric_predictions,
            metric_targets,
        )

        total_images += len(labels)

test_results = test_metric.compute()

average_inference_seconds = (
    total_inference_seconds / total_images
)

fps = 1 / average_inference_seconds

print("Test evaluation complete")
print("Images evaluated:", total_images)
print("mAP@0.5:0.95:", float(test_results["map"]))
print("mAP@0.5:", float(test_results["map_50"]))
print(
    "Average inference time:",
    average_inference_seconds,
    "seconds/image",
)
print("Approximate FPS:", fps)

Test evaluation complete
Images evaluated: 1111
mAP@0.5:0.95: 0.23586353659629822
mAP@0.5: 0.46585574746131897
Average inference time: 0.14626225436644497 seconds/image
Approximate FPS: 6.837033958840832


In [57]:
class_names = [
    "holothurian",
    "echinus",
    "scallop",
    "starfish",
]

print("Per-class AP@0.5:0.95")

for class_name, ap_value in zip(
    class_names,
    test_results["map_per_class"],
):
    print(
        class_name,
        float(ap_value),
    )

Per-class AP@0.5:0.95
holothurian 0.2180209904909134
echinus 0.3618415892124176
scallop 0.05784988775849342
starfish 0.30574169754981995


In [58]:
print(
    "Classes returned:",
    test_results["classes"],
)

Classes returned: tensor([0, 1, 2, 3], dtype=torch.int32)


In [59]:
import pandas as pd

detr_summary = pd.DataFrame(
    [
        {
            "model": "DETR ResNet-50",
            "dataset": "DUO test",
            "best_epoch": 12,
            "mAP_50": float(test_results["map_50"]),
            "mAP_50_95": float(test_results["map"]),
            "average_inference_seconds": average_inference_seconds,
            "fps": fps,
            "test_images": total_images,
        }
    ]
)

detr_summary_path = (
    RUN_DIR / "detr_test_summary.csv"
)

detr_summary.to_csv(
    detr_summary_path,
    index=False,
)

display(detr_summary)

print("Saved to:", detr_summary_path)

,model,dataset,best_epoch,mAP_50,mAP_50_95,average_inference_seconds,fps,test_images
0,DETR ResNet-50,DUO test,12,0.465856,0.235864,0.146262,6.837034,1111


Saved to: C:\Users\megdo\Desktop\underwater-object-detection\results\detr\duo_detr_resnet50_baseline_corrected\detr_test_summary.csv


In [60]:
detr_per_class = pd.DataFrame(
    {
        "class_id": test_results["classes"].cpu().numpy(),
        "class_name": class_names,
        "AP_50_95": [
            float(value)
            for value in test_results["map_per_class"]
        ],
    }
)

detr_per_class_path = (
    RUN_DIR / "detr_per_class_results.csv"
)

detr_per_class.to_csv(
    detr_per_class_path,
    index=False,
)

display(detr_per_class)

print("Saved to:", detr_per_class_path)

,class_id,class_name,AP_50_95
0,0,holothurian,0.218021
1,1,echinus,0.361842
2,2,scallop,0.057850
3,3,starfish,0.305742


Saved to: C:\Users\megdo\Desktop\underwater-object-detection\results\detr\duo_detr_resnet50_baseline_corrected\detr_per_class_results.csv
